In [ ]:
video_path = 
output_video_path = 
raw_ml2_df = pd.read_csv()

In order to align videos with data better, there are three paramters that can be played with:

1- frames_to_skip

2- frame_min_x, frame_max_x

3- frame_min_y, frame_max_y

In [ ]:
# Get metadata of video

total_frames = 0
original_width, original_height = 1280, 720
frame_width = 0
frame_height = 0

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    # Get video properties
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = total_frames / fps  # Duration in seconds

    # Print video metadata
    print(f"Total Frames: {total_frames}")
    print(f"Resolution: {frame_width}x{frame_height}")
    print(f"FPS: {fps}")
    print(f"Duration: {duration:.2f} seconds")

# Release resources
cap.release()

In [ ]:
experiment_ml2_df = raw_ml2_df.loc[(raw_ml2_df[" QRDetection Status"] == False) & (raw_ml2_df[" Hardcoded Eye Gaze"] == True)]

experiment_ml2_df = experiment_ml2_df.drop_duplicates(subset = ["TimeStamp", " frame #"], keep = "first")

In [ ]:
experiment_ml2_df = experiment_ml2_df[experiment_ml2_df[" DetectedFaces Info"] != "{}"]
number_of_exp_rows = experiment_ml2_df.shape[0]

In [ ]:
def extractTwoDimPos(jsonObj):

  if jsonObj == "{}":
    return

  replacedCommas = jsonObj.replace("|", ",")

  stringLength = len(replacedCommas)
  replaceIndex = stringLength - 2
  replacedCommas = replacedCommas[:replaceIndex] + " " + replacedCommas[replaceIndex + 1:]

  json_object = json.loads(replacedCommas)

  for faceCube in json_object:
    twoDimPos = eval(json_object[faceCube]["2D Position"])

    if not faceCubesAndTwoDimPos:
      faceCubesAndTwoDimPos[faceCube] = [twoDimPos]
    else:
      if faceCube not in faceCubesAndTwoDimPos:

        number_of_frames_elapsed = max(len(value) for value in faceCubesAndTwoDimPos.values())
        prepend_tuples_list = [(0, 0)] * (number_of_frames_elapsed - 1)

        faceCubesAndTwoDimPos[faceCube] = prepend_tuples_list
        faceCubesAndTwoDimPos[faceCube].append(twoDimPos)

      else:
        faceCubesAndTwoDimPos[faceCube].append(twoDimPos)

  faceCubes_in_row = list(json_object.keys())
  for key in faceCubesAndTwoDimPos.keys():
    if key not in faceCubes_in_row:
      faceCubesAndTwoDimPos[key].append((0,0))

In [ ]:
def tupleListToXYLists(tupleList):
  xList = []
  yList = []

  for tuple in tupleList:
    xList.append(tuple[0])
    yList.append(tuple[1])
      
  xList = list(set(xList))
  yList = list(set(yList))

  return xList, yList

In [ ]:
faceCubesAndTwoDimPos = {}
experiment_ml2_df[" DetectedFaces Info"].apply(extractTwoDimPos)

for key, value in faceCubesAndTwoDimPos.items():
    print(key)

In [ ]:
scale_x, scale_y = None, None

In [ ]:
frame_counter = 0

In [ ]:
data_min_x, data_max_x = float("inf"), float("-inf")
data_min_y, data_max_y = float("inf"), float("-inf")

for faceCube in faceCubesAndTwoDimPos.keys():
    for frame_data in faceCubesAndTwoDimPos[faceCube]:
        x, y = frame_data

        if x == 0 and y == 0:
            continue
            
        data_min_x = min(data_min_x, x)
        data_max_x = max(data_max_x, x)
        data_min_y = min(data_min_y, y)
        data_max_y = max(data_max_y, y)

print("data_min_x: ", data_min_x)
print("data_max_x: ", data_max_x)
print("data_min_y: ", data_min_y)
print("data_max_y: ", data_max_y)

In [ ]:
frame_min_x, frame_max_x = 100, 1900
frame_min_y, frame_max_y = 700, 800

In [ ]:
def scale_x_value(value):
    return int(frame_min_x + ((value - data_min_x) / (data_max_x - data_min_x)) * (frame_max_x - frame_min_x))

def scale_y_value(value):
    return int(frame_min_y + ((value - data_min_y) / (data_max_y - data_min_y)) * (frame_max_y - frame_min_y))

In [ ]:
frame_counter = 0
trajectories = {}
colors = {}

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frames_to_skip)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(1000 / fps)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    cv2.namedWindow("Video Playback", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Video Playback", frame_width, frame_height)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened() and frame_counter < number_of_exp_rows:
        ret, frame = cap.read()
        if not ret:
            break

        for faceCube in faceCubesAndTwoDimPos.keys():
            curr_faceCube_pos = faceCubesAndTwoDimPos[faceCube][frame_counter]

            x = int(curr_faceCube_pos[0])
            y = int(curr_faceCube_pos[1])

            if x != 0 or y != 0:
                x = scale_x_value(x)
                center = (x, y)

                if faceCube not in trajectories:
                    trajectories[faceCube] = [] 
                    colors[faceCube] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

                cv2.circle(frame, center, 10, colors[faceCube], -1) 
                trajectories[faceCube].append(center)  
            else:
                if faceCube not in trajectories:
                    trajectories[faceCube] = []  
                    colors[faceCube] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
                trajectories[faceCube].append(None) 

            points = trajectories[faceCube]
            for i in range(1, len(points)):
                if points[i - 1] is None or points[i] is None:
                    continue  
                cv2.line(frame, points[i - 1], points[i], colors[faceCube], 1)

        legend_x = 10
        legend_y = 10
        font_scale = 0.5
        font_thickness = 1
        box_height = 20
        box_width = 150
        y_offset = 0

        for faceCube, color in colors.items():
            top_left = (legend_x, legend_y + y_offset)
            bottom_right = (legend_x + box_width, legend_y + box_height + y_offset)

            cv2.rectangle(frame, top_left, bottom_right, (50, 50, 50), -1)

            color_box_top_left = (legend_x + 5, legend_y + 5 + y_offset)
            color_box_bottom_right = (legend_x + 25, legend_y + box_height - 5 + y_offset)
            cv2.rectangle(frame, color_box_top_left, color_box_bottom_right, color, -1)

            text_position = (legend_x + 30, legend_y + 15 + y_offset)
            cv2.putText(frame, f"{faceCube}", text_position, cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), font_thickness)

            y_offset += box_height + 5

        out.write(frame)

        cv2.imshow("Video Playback", frame)
        frame_counter += 1

        if cv2.waitKey(frame_interval) & 0xFF == ord('q'):
            break

cap.release()
out.release()
cv2.destroyAllWindows()